# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FatimaNdeem/Flyrank-ml-internship./blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import subprocess
import os

subprocess.run([
    "git", "clone",
    "https://github.com/FatimaNdeem/Flyrank-ml-internship.",
    "/content/Flyrank-ml-internship."
])

os.chdir("/content/Flyrank-ml-internship.")
print("Current directory:", os.getcwd())
print("Dataset exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Current directory: /content/Flyrank-ml-internship.
Dataset exists: True


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I will build the feature vector from signals that are available before the refresh decision. The initial feature set uses recent search performance, content freshness, search demand, content characteristics, and search-position signals. Numerical missing values will be filled using the median of the corresponding feature, while categorical missing values will be filled with "Unknown" and then encoded. I will exclude identifiers and any fields that are derived from the outcome or contain future information

In [ ]:
# Select features for the refresh opportunity vector

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

# Keep only the selected columns
X = df[numeric_features + categorical_features].copy()

# Fill numerical missing values with the median
for col in numeric_features:
    X[col] = X[col].fillna(X[col].median())

# Fill categorical missing values
for col in categorical_features:
    X[col] = X[col].fillna("Unknown")

# One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

print("Feature vector shape:", X.shape)
print("Missing values remaining:", X.isna().sum().sum())
print("\nFirst 5 rows:")
X.head()

Feature vector shape: (30000, 67)
Missing values remaining: 0

First 5 rows:


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,char_count_tier_Unknown,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,22,17,16,...,0,0,1,0,0,0,0,0,1,0
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,9,...,0,0,1,0,0,0,0,1,0,0
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,14,11,11,...,0,0,1,0,0,0,0,1,0,0
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,87,78,75,...,1,0,1,0,0,0,1,0,0,0
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,177,145,144,...,0,0,1,0,0,0,0,1,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The numeric features describe search demand, content size, historical performance, recent performance, content age, freshness, and engagement. Missing numeric values are filled with the median of the available values.

The categorical features describe competition level, content type, search intent, age, freshness, content size, impression level, and position tier. Missing categorical values are filled with "Unknown" and then one-hot encoded.

These features are intended to represent information available before the refresh decision. The historical and recent performance fields describe observed performance windows, while content age, update timing, and content characteristics are available before the decision. The feature vector does not include the future outcome used to evaluate whether the content actually declined later.

In [ ]:
# Check feature types and missing values

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Final feature count:", X.shape[1])
print("Missing values:", X.isna().sum().sum())

feature_notes_check = pd.DataFrame({
    "feature_group": ["numeric", "categorical"],
    "count": [len(numeric_features), len(categorical_features)],
    "missing_handling": ["median", "Unknown + one-hot encoding"]
})

feature_notes_check

Numeric features: 28
Categorical features: 9
Final feature count: 67
Missing values: 0


,feature_group,count,missing_handling
0,numeric,28,median
1,categorical,9,Unknown + one-hot encoding


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage hunt

I checked the selected features for fields that directly represent the future outcome or are derived from the target. I also checked for future-window features and product-related flags. The selected feature vector uses observed historical and recent performance measures, content characteristics, and freshness information. I do not use a future performance label or a later outcome as an input feature.

In [ ]:
# Leakage checks

label_like = [
    "is_declining",
    "label",
    "target",
    "outcome",
    "action_label",
    "reason_code",
    "action_score"
]

future_like = [
    "future",
    "next_30d",
    "next_60d",
    "next_90d"
]

product_like = [
    "product",
    "product_flag",
    "product_type"
]

selected_original_features = numeric_features + categorical_features

label_matches = [
    col for col in selected_original_features
    if any(term in col.lower() for term in label_like)
]

future_matches = [
    col for col in selected_original_features
    if any(term in col.lower() for term in future_like)
]

product_matches = [
    col for col in selected_original_features
    if any(term in col.lower() for term in product_like)
]

print("Label-derived matches:", label_matches)
print("Future-window matches:", future_matches)
print("Product-flag matches:", product_matches)

print("\nLeakage check passed:",
      len(label_matches) == 0
      and len(future_matches) == 0
      and len(product_matches) == 0)

Label-derived matches: []
Future-window matches: []
Product-flag matches: []

Leakage check passed: True


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

What I excluded and why

I excluded content_id and client_id because they are identifiers and could allow the model to memorize individual records or clients rather than learn generalizable patterns.

I excluded provider_used and model_used because they describe the system/provider used to generate data and are not necessary for the refresh-opportunity decision.

I excluded trend_direction and trend_pct because they summarize the same recent performance change that could be used to define the decline/opportunity target. Including them could make the model rely directly on a target-related signal.

I excluded any future-window performance fields and any fields representing a final decision, action, label, reason code, or priority score because these would not be independent inputs available at prediction time.

In [ ]:
# Fields excluded from the feature vector

excluded_features = {
    "content_id": "Identifier; does not describe measurable content characteristics.",
    "client_id": "Identifier; excluded to avoid client-specific memorization.",
    "provider_used": "System/provider field; not needed for the refresh-opportunity question.",
    "model_used": "System/model field; not needed for the refresh-opportunity question.",
    "trend_direction": "Derived from performance trend; closely tied to the decline signal.",
    "trend_pct": "Direct performance-trend measure; excluded to avoid target leakage.",
    "reason_code": "Baseline decision output, not an independent input.",
    "action_label": "Baseline action output, not an independent input.",
    "action_score": "Baseline score output, not an independent input."
}

excluded_check = pd.DataFrame(
    list(excluded_features.items()),
    columns=["excluded_feature", "reason"]
)

print("Number of excluded fields:", len(excluded_check))
excluded_check

Number of excluded fields: 9


,excluded_feature,reason
0,content_id,Identifier; does not describe measurable conte...
1,client_id,Identifier; excluded to avoid client-specific ...
2,provider_used,System/provider field; not needed for the refr...
3,model_used,System/model field; not needed for the refresh...
4,trend_direction,Derived from performance trend; closely tied t...
5,trend_pct,Direct performance-trend measure; excluded to ...
6,reason_code,"Baseline decision output, not an independent i..."
7,action_label,"Baseline action output, not an independent input."
8,action_score,"Baseline score output, not an independent input."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.